# map of wroclaw, points of laboratories and hospital

* pkty, szpital i droga

In [18]:
from pathlib import Path
import pandas as pd
import folium
import osmnx as ox

# ============================================================
# 1. USTAWIENIA
# ============================================================

OUT_DIR = Path("/Users/weron/Desktop/wroclaw_map_output")
OUT_DIR.mkdir(exist_ok=True)

# ============================================================
# 2. PUNKTY
#    Na razie ręcznie wpisane współrzędne przykładowe/robocze.
#    Potem możesz je podmienić na finalne.
# ============================================================

# to sa losowe pkty we wroclawiu
points = [
    {"name": "Central Hospital", "lat": 51.0730, "lon": 17.0740, "type": "hospital"},
    {"name": "Lab 1", "lat": 51.1095, "lon": 17.0328, "type": "lab"},
    {"name": "Lab 2", "lat": 51.0937, "lon": 16.9988, "type": "lab"},
    {"name": "Lab 3", "lat": 51.1160, "lon": 16.9705, "type": "lab"},
    {"name": "Lab 4", "lat": 51.0620, "lon": 16.9950, "type": "lab"},
    {"name": "Lab 5", "lat": 51.0485, "lon": 16.9780, "type": "lab"},
    {"name": "Lab 6", "lat": 51.0915, "lon": 17.0615, "type": "lab"},
    {"name": "Lab 7", "lat": 51.0955, "lon": 17.0455, "type": "lab"},
    {"name": "Lab 8", "lat": 51.1180, "lon": 16.9340, "type": "lab"},
    {"name": "Lab 9", "lat": 51.1135, "lon": 16.9040, "type": "lab"},
    {"name": "Lab 10", "lat": 51.1000, "lon": 17.0260, "type": "lab"},
    {"name": "Lab 11", "lat": 51.0525, "lon": 17.0070, "type": "lab"},
    {"name": "Lab 12", "lat": 51.0615, "lon": 16.9910, "type": "lab"},
]


# realne adresy-> trzeba je przepisac na dokladne wspolrzedne 
# points = [
#     {"name": "Central Hospital", "address": "ul. Borowska 213, Wrocław, Poland", "type": "hospital"},
#     {"name": "Lab 1", "address": "ul. Krawiecka 3C, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 2", "address": "ul. Jana Sebastiana Bacha 13, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 3", "address": "ul. Kamiennogórska 10, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 4", "address": "ul. Cedrowa 4, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 5", "address": "ul. Czekoladowa 49A, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 6", "address": "ul. Fieldorfa 2, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 7", "address": "ul. Dąbrowskiego 44, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 8", "address": "ul. Hermanowska 99A, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 9", "address": "ul. Krępicka 1, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 10", "address": "ul. Swobodna 8A, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 11", "address": "ul. Zwycięska 14CA/A, Wrocław, Poland", "type": "lab"},
#     {"name": "Lab 12", "address": "ul. Życzliwa 17, Wrocław, Poland", "type": "lab"},
# ]



df = pd.DataFrame(points)
df.to_csv(OUT_DIR / "points_manual.csv", index=False, encoding="utf-8-sig")

# ============================================================
# 3. POBRANIE SIECI DRÓG WROCŁAWIA
#    network_type="drive" = drogi przejezdne dla samochodów
# ============================================================

print("Pobieram sieć drogową Wrocławia...")
G = ox.graph_from_place("Wrocław, Poland", network_type="drive")

# Zamiana grafu na GeoDataFrame
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

print("Liczba węzłów:", len(nodes_gdf))
print("Liczba odcinków dróg:", len(edges_gdf))

# ============================================================
# 4. MAPA
# ============================================================

center_lat = df["lat"].mean()
center_lon = df["lon"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="OpenStreetMap"
)

# ============================================================
# 5. RYSOWANIE DRÓG
# ============================================================

for _, edge in edges_gdf.iterrows():
    geom = edge.geometry
    if geom is None:
        continue

    coords = [(y, x) for x, y in geom.coords]

    folium.PolyLine(
        locations=coords,
        weight=1,
        opacity=0.25
    ).add_to(m)

# ============================================================
# 6. RYSOWANIE PUNKTÓW
# ============================================================

for idx, row in df.iterrows():
    if row["type"] == "hospital":
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=9,
            popup=f"<b>{row['name']}</b>",
            tooltip=row["name"],
            color="red",
            fill=True,
            fill_opacity=1.0
        ).add_to(m)
    else:
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6,
            popup=f"{row['name']}",
            tooltip=row["name"],
            color="blue",
            fill=True,
            fill_opacity=0.9
        ).add_to(m)

        # numer obok punktu
        folium.Marker(
            [row["lat"], row["lon"]],
            icon=folium.DivIcon(
                html=f"""
                <div style="font-size: 10pt; color: black; font-weight: bold;">
                    {idx}
                </div>
                """
            )
        ).add_to(m)

# ============================================================
# 7. ZAPIS
# ============================================================

map_path = OUT_DIR / "wroclaw_roads_points.html"
m.save(map_path)

print("Gotowe.")
print("Mapa:", map_path.resolve())
print("Punkty:", (OUT_DIR / "points_manual.csv").resolve())

Pobieram sieć drogową Wrocławia...
Liczba węzłów: 7291
Liczba odcinków dróg: 17213
Gotowe.
Mapa: C:\Users\weron\Desktop\wroclaw_map_output\wroclaw_roads_points.html
Punkty: C:\Users\weron\Desktop\wroclaw_map_output\points_manual.csv


* pkty szpital i droga z jednego lab do szpitala

In [19]:
import networkx as nx
import osmnx as ox

# ============================================================
# 1. WYBIERAMY SZPITAL I JEDNO LABORATORIUM
# ============================================================

hospital = df[df["type"] == "hospital"].iloc[0]
lab = df[df["type"] == "lab"].iloc[0]  # pierwsze lab

print("Hospital:", hospital["name"])
print("Lab:", lab["name"])

# ============================================================
# 2. ZNAJDUJEMY NAJBLIŻSZE WĘZŁY W GRAFIE
# ============================================================

hospital_node = ox.distance.nearest_nodes(G, hospital["lon"], hospital["lat"])
lab_node = ox.distance.nearest_nodes(G, lab["lon"], lab["lat"])

# ============================================================
# 3. NAJKRÓTSZA TRASA (PO DROGACH!)
# ============================================================

route = nx.shortest_path(G, hospital_node, lab_node, weight="length")

# długość trasy
route_length = nx.shortest_path_length(G, hospital_node, lab_node, weight="length")

print(f"Długość trasy: {route_length/1000:.2f} km")

# ============================================================
# 4. KONWERSJA NA WSPÓŁRZĘDNE
# ============================================================

route_coords = [(G.nodes[node]["y"], G.nodes[node]["x"]) for node in route]

# ============================================================
# 5. MAPA Z TRASĄ
# ============================================================

m_route = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=12
)

# punkty
for _, row in df.iterrows():
    color = "red" if row["type"] == "hospital" else "blue"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7,
        color=color,
        fill=True
    ).add_to(m_route)

# trasa
folium.PolyLine(
    locations=route_coords,
    weight=5,
    opacity=0.9,
    tooltip="Route hospital → lab"
).add_to(m_route)

# zapis
route_map_path = OUT_DIR / "route_example.html"
m_route.save(route_map_path)

print(f"Mapa trasy zapisana: {route_map_path.resolve()}")

Hospital: Central Hospital
Lab: Lab 1
Długość trasy: 5.85 km
Mapa trasy zapisana: C:\Users\weron\Desktop\wroclaw_map_output\route_example.html


* datasets

In [ ]:
import pandas as pd

hospital = {
    "node_id": 0,
    "name": "Central Hospital",
    "type": "hospital",
    "lat": 51.0730,
    "lon": 17.0740
}


# ZASTAPIMY PRAWDZIWYMI WSPOLRZEDNYMI 
labs = [
    {"node_id": 1, "name": "Lab 1", "type": "lab", "lat": 51.1095, "lon": 17.0328},
    {"node_id": 2, "name": "Lab 2", "type": "lab", "lat": 51.0937, "lon": 16.9988},
    {"node_id": 3, "name": "Lab 3", "type": "lab", "lat": 51.1160, "lon": 16.9705},
    {"node_id": 4, "name": "Lab 4", "type": "lab", "lat": 51.0620, "lon": 16.9950},
    {"node_id": 5, "name": "Lab 5", "type": "lab", "lat": 51.0485, "lon": 16.9780},
    {"node_id": 6, "name": "Lab 6", "type": "lab", "lat": 51.0915, "lon": 17.0615},
    {"node_id": 7, "name": "Lab 7", "type": "lab", "lat": 51.0955, "lon": 17.0455},
    {"node_id": 8, "name": "Lab 8", "type": "lab", "lat": 51.1180, "lon": 16.9340},
    {"node_id": 9, "name": "Lab 9", "type": "lab", "lat": 51.1135, "lon": 16.9040},
    {"node_id": 10, "name": "Lab 10", "type": "lab", "lat": 51.1000, "lon": 17.0260},
    {"node_id": 11, "name": "Lab 11", "type": "lab", "lat": 51.0525, "lon": 17.0070},
    {"node_id": 12, "name": "Lab 12", "type": "lab", "lat": 51.0615, "lon": 16.9910},
]

requests = [
    {"request_id": 1, "lab_node_id": 1, "demand": 14, "ready_time": 30, "due_time": 90, "service_time": 5, "max_transport_time": 120},
    {"request_id": 2, "lab_node_id": 2, "demand": 10, "ready_time": 45, "due_time": 100, "service_time": 5, "max_transport_time": 150},
    {"request_id": 3, "lab_node_id": 3, "demand": 18, "ready_time": 20, "due_time": 80, "service_time": 5, "max_transport_time": 120},
    {"request_id": 4, "lab_node_id": 4, "demand": 9, "ready_time": 35, "due_time": 70, "service_time": 5, "max_transport_time": 75},
]

vehicles = [
    {"vehicle_id": 1, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 300},
    {"vehicle_id": 2, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 300},
]

nodes_df = pd.DataFrame([hospital] + labs)
requests_df = pd.DataFrame(requests)
vehicles_df = pd.DataFrame(vehicles)

nodes_df.to_csv("/Users/weron/Desktop/wroclaw_map_output/nodes.csv", index=False, encoding="utf-8-sig")
requests_df.to_csv("/Users/weron/Desktop/wroclaw_map_output/requests.csv", index=False, encoding="utf-8-sig")
vehicles_df.to_csv("/Users/weron/Desktop/wroclaw_map_output/vehicles.csv", index=False, encoding="utf-8-sig")

* macierz czasu przejazdu miedzy wezlami 
(nastepny krok)

In [23]:
from pathlib import Path
import pandas as pd
import osmnx as ox
import networkx as nx
import numpy as np

# ============================================================
# 1. ŚCIEŻKI
# ============================================================

OUT_DIR.mkdir(exist_ok=True)

nodes_path = OUT_DIR / "nodes.csv"

# ============================================================
# 2. WCZYTANIE DANYCH
# ============================================================

nodes_df = pd.read_csv(nodes_path)

print("Wczytane węzły:")
print(nodes_df[["node_id", "name", "type", "lat", "lon"]])

# ============================================================
# 3. POBRANIE GRAFU I DODANIE CZASÓW PRZEJAZDU
# ============================================================

print("Pobieram graf drogowy...")
G = ox.graph_from_place("Wrocław, Poland", network_type="drive")

# Dodanie prędkości i czasu przejazdu do krawędzi
G = ox.routing.add_edge_speeds(G)
G = ox.routing.add_edge_travel_times(G)

print("Graf gotowy.")

# ============================================================
# 4. PRZYPISANIE PUNKTÓW DO NAJBLIŻSZYCH WĘZŁÓW GRAFU
# ============================================================

# Uwaga: X = longitude, Y = latitude
nearest_nodes = ox.distance.nearest_nodes(
    G,
    X=nodes_df["lon"].values,
    Y=nodes_df["lat"].values
)

nodes_df["osmnx_node"] = nearest_nodes

nodes_df.to_csv(OUT_DIR / "nodes_with_osm_ids.csv", index=False, encoding="utf-8-sig")
print("Zapisano nodes_with_osm_ids.csv")

# ============================================================
# 5. MACIERZ CZASÓW PRZEJAZDU
# ============================================================

node_ids = nodes_df["node_id"].tolist()
names = nodes_df["name"].tolist()
osm_nodes = nodes_df["osmnx_node"].tolist()

n = len(nodes_df)

time_matrix = pd.DataFrame(
    data=np.full((n, n), np.nan),
    index=node_ids,
    columns=node_ids
)

distance_matrix = pd.DataFrame(
    data=np.full((n, n), np.nan),
    index=node_ids,
    columns=node_ids
)

print("Liczę macierz czasów przejazdu...")

for i in range(n):
    source_osm = osm_nodes[i]

    # najkrótsze czasy od jednego źródła do wszystkich
    travel_times = nx.single_source_dijkstra_path_length(
        G,
        source_osm,
        weight="travel_time"
    )

    distances = nx.single_source_dijkstra_path_length(
        G,
        source_osm,
        weight="length"
    )

    for j in range(n):
        target_osm = osm_nodes[j]

        if target_osm in travel_times:
            # travel_time w sekundach -> minuty
            time_matrix.iloc[i, j] = travel_times[target_osm] / 60.0

        if target_osm in distances:
            # length w metrach
            distance_matrix.iloc[i, j] = distances[target_osm]

# ============================================================
# 6. ZAPIS MACIERZY
# ============================================================

time_matrix.to_csv(OUT_DIR / "travel_time_matrix_min.csv", encoding="utf-8-sig")
distance_matrix.to_csv(OUT_DIR / "distance_matrix_m.csv", encoding="utf-8-sig")

print("Zapisano:")
print(OUT_DIR / "travel_time_matrix_min.csv")
print(OUT_DIR / "distance_matrix_m.csv")

# ============================================================
# 7. DODATKOWO: WERSJA 'LONG FORMAT'
# ============================================================

rows = []
for i in range(n):
    for j in range(n):
        rows.append({
            "from_node_id": node_ids[i],
            "from_name": names[i],
            "to_node_id": node_ids[j],
            "to_name": names[j],
            "travel_time_min": time_matrix.iloc[i, j],
            "distance_m": distance_matrix.iloc[i, j],
        })

travel_long_df = pd.DataFrame(rows)
travel_long_df.to_csv(OUT_DIR / "travel_times_long.csv", index=False, encoding="utf-8-sig")

print("Zapisano także:")
print(OUT_DIR / "travel_times_long.csv")
print("Gotowe.")

Wczytane węzły:
    node_id              name      type      lat      lon
0         0  Central Hospital  hospital  51.0730  17.0740
1         1             Lab 1       lab  51.1095  17.0328
2         2             Lab 2       lab  51.0937  16.9988
3         3             Lab 3       lab  51.1160  16.9705
4         4             Lab 4       lab  51.0620  16.9950
5         5             Lab 5       lab  51.0485  16.9780
6         6             Lab 6       lab  51.0915  17.0615
7         7             Lab 7       lab  51.0955  17.0455
8         8             Lab 8       lab  51.1180  16.9340
9         9             Lab 9       lab  51.1135  16.9040
10       10            Lab 10       lab  51.1000  17.0260
11       11            Lab 11       lab  51.0525  17.0070
12       12            Lab 12       lab  51.0615  16.9910
Pobieram graf drogowy...
Graf gotowy.
Zapisano nodes_with_osm_ids.csv
Liczę macierz czasów przejazdu...
Zapisano:
\Users\weron\Desktop\wroclaw_map_output\travel_time_matrix